# Bước 3 - Information Extraction (IE) – Fact Extraction
## Module 8: Trích xuất Thực thể, Số liệu và Mốc thời gian

---

### Mục tiêu:
BM25 chỉ đo lường sự trùng lặp từ khóa, không phân biệt được sự kiện đúng hay sai về mặt thực tế.
Module 8 trích xuất 3 nhóm đặc trưng Fact quan trọng:
1. **Named Entities:** Người, Tổ chức, Địa danh, Sự kiện bằng `VnCoreNLP NER`.
2. **Temporal Facts:** Ngày, tháng, năm, thế kỷ, quý bằng Regex.
3. **Numerical Facts:** Tỷ lệ %, số lượng, đơn vị đo lường bằng Regex (kèm kỹ thuật Temporal Masking).
Từ đó tính toán:
$$\text{Fact Score} = 0.4 \times \text{Entity Match} + 0.4 \times \text{Number Match} + 0.2 \times \text{Date Match}$$
Xuất bảng kết quả vào: `outputs/evidence_features.csv`.

In [ ]:
# ======================================================================
# 1. KHAI BÁO CÁC THƯ VIỆN CẦN THIẾT
# ======================================================================
import os  # Thao tác hệ thống và thiết lập biến môi trường JAVA
import re  # Biểu thức chính quy (Regex) để trích xuất số liệu và ngày tháng
import sys  # Lấy thông tin đường dẫn môi trường Python hiện tại
import time  # Đo thời gian thực thi trích xuất đặc trưng
from pathlib import Path  # Quản lý đường dẫn file/thư mục

import pandas as pd  # Xử lý bảng dữ liệu
import py_vncorenlp  # Thư viện xử lý tiếng Việt VnCoreNLP (Word Segmentation, POS, NER)

# ======================================================================
# 2. TỰ ĐỘNG ĐỊNH VỊ THƯ MỤC GỐC VÀ CÁC ĐƯỜNG DẪN FILE
# ======================================================================
current_dir = Path.cwd().resolve()
candidates = [current_dir, current_dir.parent, current_dir.parent.parent]
PROJECT_ROOT = current_dir
for cand in candidates:
    if (cand / 'data').exists() and (cand / 'notebooks').exists():
        PROJECT_ROOT = cand
        break

# Thư mục lưu trữ kết quả của tầng Retrieval
OUTPUT_DIR = PROJECT_ROOT / 'data/processed/retrieval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# File đầu vào: Kết quả Top-5 câu ứng viên đã tìm được từ bước BM25 (Bước 2)
INPUT_PATH = OUTPUT_DIR / 'bm25_results.csv'

# ======================================================================
# 3. THIẾT LẬP MÁY ẢO JAVA (JVM) ĐỂ CHẠY VNCORENLP
# ======================================================================
# VnCoreNLP được viết bằng Java nên cần cấu hình JAVA_HOME và JVM_PATH
conda_prefix = Path(sys.prefix)
os.environ['JAVA_HOME'] = str(conda_prefix)
jvm_path = conda_prefix / 'lib' / 'server' / 'libjvm.dylib'
if jvm_path.exists():
    os.environ['JVM_PATH'] = str(jvm_path)

# Lưu lại thư mục làm việc hiện tại (vì VnCoreNLP khi khởi động có thể tự ý đổi thư mục)
orig_cwd = Path.cwd()
# Khởi tạo mô hình VnCoreNLP với 3 bộ phân tích: Tách từ (wseg), Từ loại (pos), Thực thể tên riêng (ner)
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg', 'pos', 'ner'], save_dir=str(Path.home() / '.vncorenlp'))
# Khôi phục lại thư mục làm việc ban đầu để tránh lỗi đường dẫn
os.chdir(orig_cwd)
print('✓ Khởi tạo VnCoreNLP NER thành công và bảo vệ working directory an toàn!')


2026-09-07 00:22:41 INFO  WordSegmenter:24 - Loading Word Segmentation model
2026-09-07 00:22:41 INFO  PosTagger:23 - Loading POS Tagging model
2026-09-07 00:22:44 INFO  NerRecognizer:34 - Loading NER model
✓ Khởi tạo VnCoreNLP NER thành công và bảo vệ working directory an toàn!


In [ ]:
# ======================================================================
# 1. CÁC MẪU BIỂU THỨC CHÍNH QUY CHO MỐC THỜI GIAN (TEMPORAL PATTERNS)
# ======================================================================
TEMPORAL_PATTERNS = [
    r'\b(?:ngày\s+)?\d{1,2}[\./\-]\d{1,2}[\./\-]\d{4}\b',       # Ngày dạng: 16/02/2023 hoặc 16-2-2023
    r'\b\d{1,2}\s+tháng\s+\d{1,2}(?:\s+năm\s+\d{4})?\b',      # Ngày dạng: 30 tháng 5 năm 2023
    r'\btháng\s+\d{1,2}[\./\-]\d{4}\b',                         # Tháng dạng: tháng 4/2023
    r'\btháng\s+\d{1,2}\s+năm\s+\d{4}\b',                         # Tháng dạng: tháng 4 năm 2023
    r'\bquý\s+[1-4](?:[\s/]+năm\s+\d{4}|\s+năm\s+\d{4}|\s+\d{4})?\b', # Dạng: quý 1 năm 2024
    r'\bthế\s+kỷ\s+(?:[ivxldcm]+|\d+)\b',                             # Dạng: thế kỷ 21, thế kỷ XXI
    r'\b(?:năm\s+)?(18\d{2}|19\d{2}|20\d{2})\b',                    # Năm dạng 4 chữ số: năm 1890, 2024
]

# ======================================================================
# 2. HÀM TRÍCH XUẤT MỐC THỜI GIAN (TEMPORAL FACTS)
# ======================================================================
def extract_temporal_facts(text: str) -> set[str]:
    """Quét và trả về tập hợp các mốc thời gian xuất hiện trong câu."""
    if not isinstance(text, str):
        return set()  # Trả về tập rỗng nếu dữ liệu không phải chuỗi
    dates = set()
    text_l = text.lower()  # Chuyển về chữ thường để quét không phân biệt hoa thường
    for pat in TEMPORAL_PATTERNS:
        for m in re.finditer(pat, text_l):
            dates.add(m.group().strip())  # Thêm mốc thời gian tìm được vào tập hợp
    return dates

# ======================================================================
# 3. HÀM TRÍCH XUẤT SỐ LIỆU VÀ ĐẠI LƯỢNG (NUMERICAL FACTS - KÈM MASKING)
# ======================================================================
def extract_numerical_facts(text: str) -> set[str]:
    """Trích xuất số liệu nhưng che (mask) ngày tháng trước để không bị nhầm lẫn."""
    if not isinstance(text, str):
        return set()
    text_l = text.lower()
    
    # BƯỚC 1 (TEMPORAL MASKING): Thay thế các mốc thời gian bằng khoảng trắng
    # Mẹo này giúp máy không nhầm 'năm 2024' thành số lượng 2024!
    for pat in TEMPORAL_PATTERNS:
        text_l = re.sub(pat, ' ', text_l)
        
    nums = set()
    # BƯỚC 2: Quét các số liệu dạng tỷ lệ phần trăm (ví dụ: 8.5%, 100%)
    for m in re.finditer(r'\b\d+(?:[\.,]\d+)?\s*%', text_l):
        nums.add(m.group().replace(' ', ''))
    text_l = re.sub(r'\b\d+(?:[\.,]\d+)?\s*%', ' ', text_l)  # Che phần trăm đã lấy
    
    # BƯỚC 3: Quét số liệu đi kèm đơn vị tính / tiền tệ / đơn vị đo
    units = r'(?:triệu|tỷ|nghìn|ngàn|vạn|đô|usd|eur|đồng|vnd|km|m2|m3|m|cm|kg|tấn|lít|héc-ta|ha|người|ca|bệnh nhân|bàn|điểm|tuổi|năm tù)'
    for m in re.finditer(rf'\b\d+(?:[\.,]\d+)?\s*{units}\b', text_l):
        nums.add(m.group().strip())
    text_l = re.sub(rf'\b\d+(?:[\.,]\d+)?\s*{units}\b', ' ', text_l)  # Che số kèm đơn vị
    
    # BƯỚC 4: Quét các con số thuần túy còn lại (số nguyên, số thập phân)
    for m in re.finditer(r'\b\d+(?:[\.,]\d+)?\b', text_l):
        nums.add(m.group().strip())
    return nums

# ======================================================================
# 4. HÀM BÓC TÁCH THỰC THỂ TÊN RIÊNG (NAMED ENTITY EXTRACTION)
# ======================================================================
def extract_entities_from_ann(annotated_dict: dict) -> set[str]:
    """Gom các nhãn BIO từ kết quả VnCoreNLP thành tên thực thể hoàn chỉnh."""
    ents = set()
    cur = []
    # Duyệt qua từng câu và từng từ đã được VnCoreNLP gán nhãn thực thể
    for s_idx, tokens in annotated_dict.items():
        for tok in tokens:
            ner = tok['nerLabel']  # Nhãn thực thể (ví dụ B-PER, I-PER, B-LOC, O)
            word = tok['wordForm'].replace('_', ' ').lower()  # Bỏ dấu gạch dưới nối từ ghép
            
            if ner.startswith('B-'):  # B- nghĩa là từ bắt đầu của một tên riêng
                if cur:
                    ents.add(' '.join(cur))  # Lưu thực thể trước đó nếu có
                cur = [word]  # Bắt đầu thực thể mới
            elif ner.startswith('I-') and cur:  # I- nghĩa là từ nối tiếp của tên riêng
                cur.append(word)
            else:  # O nghĩa là từ bình thường (không phải tên riêng)
                if cur:
                    ents.add(' '.join(cur))
                    cur = []
        if cur:
            ents.add(' '.join(cur))
            cur = []
    return ents

print('✓ Đã định nghĩa các hàm trích xuất Fact (Thời gian, Số liệu kèm Masking, Thực thể).')


✓ Đã định nghĩa các hàm trích xuất Fact.


In [ ]:
# ======================================================================
# 1. BÓC TÁCH ĐẶC TRƯNG HÀNG LOẠT VÀ DÙNG CACHING TĂNG TỐC
# ======================================================================
# Đọc dữ liệu bảng Top-5 câu ứng viên từ Bước 2
df_bm25 = pd.read_csv(INPUT_PATH)
# Lọc ra danh sách tất cả các câu phân biệt (không quét lặp lại câu trùng)
unique_sents = list(set(df_bm25['claim'].unique()).union(set(df_bm25['retrieved_evidence'].unique())))
print(f'• Trích xuất đặc trưng cho {len(unique_sents):,} câu phân biệt...')

feature_cache = {}  # Bộ nhớ đệm lưu kết quả trích xuất của từng câu
t0 = time.time()
for s in unique_sents:
    ann = rdrsegmenter.annotate_text(s)  # Chạy mô hình VnCoreNLP
    feature_cache[s] = {
        'entities': extract_entities_from_ann(ann),  # Bóc tách tên riêng
        'dates': extract_temporal_facts(s),          # Bóc tách ngày tháng
        'numbers': extract_numerical_facts(s)        # Bóc tách con số
    }
os.chdir(orig_cwd)  # Khôi phục thư mục làm việc
print(f'✓ Hoàn tất trích xuất đặc trưng trong {time.time() - t0:.2f}s!')

# ======================================================================
# 2. HÀM TÍNH ĐỘ TRÙNG KHỚP DỮ KIỆN (MATCH SCORE)
# ======================================================================
def match_score(c_set: set, e_set: set, ev_text: str) -> float:
    """Tính tỷ lệ % dữ kiện trong câu Tuyên bố xuất hiện trong Bằng chứng."""
    if not c_set:
        return 0.5  # Gán giá trị trung tính nếu Tuyên bố không chứa loại dữ kiện này
    ev_l = ev_text.lower()
    # Đếm số lượng dữ kiện của Tuyên bố có mặt trong Bằng chứng
    hits = sum(1 for x in c_set if x in e_set or x in ev_l)
    return hits / len(c_set)  # Trả về điểm số trong đoạn [0, 1]

# ======================================================================
# 3. TÍNH FACT SCORE TỔNG HỢP THEO TỶ LỆ 40% - 40% - 20%
# ======================================================================
ent_m, num_m, date_m, fact_s = [], [], [], []
for _, row in df_bm25.iterrows():
    c = feature_cache[row['claim']]  # Lấy đặc trưng câu Tuyên bố
    e = feature_cache[row['retrieved_evidence']]  # Lấy đặc trưng câu Bằng chứng máy tìm
    ev = row['retrieved_evidence']
    
    em = match_score(c['entities'], e['entities'], ev)  # Độ trùng Thực thể
    nm = match_score(c['numbers'], e['numbers'], ev)    # Độ trùng Số liệu
    dm = match_score(c['dates'], e['dates'], ev)        # Độ trùng Thời gian
    
    # Công thức cốt lõi: 40% Thực thể + 40% Số liệu + 20% Mốc thời gian
    fs = 0.4 * em + 0.4 * nm + 0.2 * dm
    
    ent_m.append(round(em, 4))
    num_m.append(round(nm, 4))
    date_m.append(round(dm, 4))
    fact_s.append(round(fs, 4))

# Lưu các điểm số thành cột mới trong bảng dữ liệu
df_bm25['entity_match'] = ent_m
df_bm25['number_match'] = num_m
df_bm25['date_match'] = date_m
df_bm25['fact_score'] = fact_s

# Xuất kết quả ra file CSV chuẩn bị cho Bước 4 (Xếp hạng lại)
out_feat_path = OUTPUT_DIR / 'evidence_features.csv'
df_bm25.to_csv(out_feat_path, index=False)
print(f'✓ Đã lưu file đặc trưng Fact ({len(df_bm25):,} dòng) tại: {out_feat_path.name}')
display(df_bm25[['claim', 'retrieved_evidence', 'entity_match', 'number_match', 'date_match', 'fact_score']].head(5))


• Trích xuất đặc trưng cho 3,757 câu phân biệt...


✓ Hoàn tất trích xuất đặc trưng trong 7.45s!
✓ Đã lưu file: evidence_features.csv


,claim,retrieved_evidence,entity_match,number_match,date_match,fact_score
0,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...,1.0000,0.5,0.0,0.6000
1,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Trong số đó, có vua hề Charlie Chaplin (vua hề...",0.3333,0.5,0.0,0.3333
2,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Swore Oath ở khách sạn Saigon Morin năm 2004 T...,0.3333,0.5,0.0,0.3333
3,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Phu nhân cựu Tổng thống Pháp, bà Bernadette Ch...",0.0000,0.5,0.0,0.2000
4,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Saigon Morin, khách sạn 4 sao hàng đầu tại Huế...",0.0000,0.5,0.0,0.2000
